# RNN (SimpleRNN)

Owner: _(name)_

Rules shared by all three model notebooks:

- read the data and the split from `src` (never re-download, never re-split);
- fit on train, choose settings on validation, touch test once at the end;
- save test predictions with `common.save_predictions("rnn", test, predictions)`.

### Setup (same cell in every notebook)

Runs in Colab and in Cursor. In Colab it clones the repository first; for a private repository
paste a GitHub token when asked (Settings > Developer settings > Fine-grained tokens, read access
to this repository is enough).

In [ ]:
import sys, pathlib

REPO_URL = "https://github.com/OWNER/sp500-forecasting.git"   # <- set once after the repo exists

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB and not pathlib.Path("sp500-forecasting").exists() and not pathlib.Path("src/config.py").exists():
    from getpass import getpass
    token = getpass("GitHub token (leave empty if the repo is public): ")
    url = REPO_URL.replace("https://", f"https://{token}@") if token else REPO_URL
    import subprocess
    subprocess.run(["git", "clone", "--quiet", url], check=True)
if IN_COLAB and pathlib.Path("sp500-forecasting").exists():
    import os
    os.chdir("sp500-forecasting")

root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "src" / "config.py").exists())
sys.path.insert(0, str(root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src import config, common

df = common.load_prices()
print(root)
print(df.index.min().date(), "to", df.index.max().date(), "|", len(df), "trading days")
df.tail()

## 1. Train / validation / test, then windows of the last `config.WINDOW` days

In [ ]:
y = common.target_series(df)
train, val, test = common.split(y)
W = config.WINDOW

from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler(feature_range=(-1, 1))
scaler.fit(train.to_numpy().reshape(-1, 1))          # fit on train only, never on validation or test

def scale(s):
    return scaler.transform(s.to_numpy().reshape(-1, 1)).ravel()

def make_windows(scaled_values, index, prefix):
    """Inputs: the W values before day t (prefix supplies history for the first days). Target: day t."""
    full = np.concatenate([prefix, scaled_values])
    X, Y = [], []
    for i in range(len(scaled_values)):
        j = len(prefix) + i
        X.append(full[j - W:j]); Y.append(full[j])
    return np.array(X)[..., None], np.array(Y), index

train_s, val_s, test_s = scale(train), scale(val), scale(test)
X_train, Y_train, _ = make_windows(train_s[W:], train.index[W:], train_s[:W])
X_val, Y_val, _ = make_windows(val_s, val.index, train_s[-W:])
X_test, Y_test, _ = make_windows(test_s, test.index, np.concatenate([train_s, val_s])[-W:])
X_train.shape, X_val.shape, X_test.shape

## 2. The network

In [ ]:
import keras
from keras import layers
keras.utils.set_random_seed(config.RANDOM_SEED)

model = keras.Sequential([
    keras.Input(shape=(W, 1)),
    layers.SimpleRNN(32, activation="tanh"),
    layers.Dropout(0.1),
    layers.Dense(1),                      # one number out, no activation: a return can be any value
])
model.compile(optimizer="adam", loss="mean_squared_error", metrics=["mean_absolute_error"])
model.summary()

## 3. Train, watching the validation curve

In [ ]:
history = model.fit(X_train, Y_train, epochs=50, batch_size=32, validation_data=(X_val, Y_val), verbose=0)
h = history.history
plt.plot(h["val_mean_absolute_error"], label="validation MAE (scaled)"); plt.plot(h["mean_absolute_error"], label="training MAE (scaled)")
plt.xlabel("epoch"); plt.legend(); plt.title("pick the epoch count from this curve, then refit if needed")

## 4. Test once, in the original units, and save

In [ ]:
pred_scaled = model.predict(X_test, verbose=0).ravel()
test_pred = pd.Series(scaler.inverse_transform(pred_scaled.reshape(-1, 1)).ravel(), index=test.index)
print("test MAE", round(common.mae(test, test_pred), 6), "RMSE", round(common.rmse(test, test_pred), 6))
common.save_predictions("rnn", test, test_pred)